# Day 11 – Cleaned Company Employee Dataset

## Data Cleaning with Python and Pandas

This notebook loads the provided messy company employee dataset, identifies data-quality problems, applies appropriate cleaning techniques, compares the dataset before and after cleaning, and exports the final cleaned dataset.

### Cleaning workflow
1. Inspect the original dataset
2. Identify and quantify missing values
3. Detect inconsistent categorical entries
4. Check data types and convert where appropriate
5. Detect and quantify duplicate records
6. Handle missing values using suitable imputation methods
7. Standardize inconsistent categorical values
8. Remove duplicate records
9. Verify the cleaned dataset
10. Export the cleaned CSV


### Methods used

The Day 11 notes introduce missing-value detection with `isnull()`, quantifying missing values with `sum()`, deletion with `dropna()`, imputation with `fillna()` using mean/median/mode, forward filling with `ffill()`, and duplicate detection/removal with `duplicated()` and `drop_duplicates()`. The notes also distinguish deletion from imputation and emphasize choosing the method appropriately. fileciteturn4file0L3-L8

For this employee dataset, median imputation is used for numeric fields and mode imputation for categorical fields, while exact duplicate rows are removed. Categorical inconsistencies are standardized before the final verification.


In [1]:
import pandas as pd
import numpy as np
import os

file_name = "Day11_Messy_Company_Employee_Dataset.csv"

# If the CSV is not already available in Colab, upload it.
if not os.path.exists(file_name):
    from google.colab import files
    uploaded = files.upload()
    file_name = next(iter(uploaded))

df = pd.read_csv(file_name)
print("Dataset loaded successfully.")
print("Original shape:", df.shape)


Dataset loaded successfully.
Original shape: (157, 12)


## 1. Inspect the Original Dataset

In [2]:
print("First 5 rows:")
display(df.head())

print("Last 5 rows:")
display(df.tail())

print("Random sample:")
display(df.sample(5, random_state=42))

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)


First 5 rows:


,Employee_ID,Employee_Name,Department,Job_Title,Age,Gender,Annual_Salary,Experience_Years,Joining_Date,City,Performance_Score,Work_Mode
0,EMP0098,Ananya Nair,NaN,Data Scientist,48.0,Other,108371.0,13.2,2017-06-13,Pune,5.0,Office
1,EMP0046,Faizan Khan,Marketing,Marketing Manager,31.0,Female,108824.0,12.1,2021-09-15,Mumbai,2.0,Office
2,EMP0017,Saira Malik,Human Resources,HR Manager,28.0,Female,119400.0,13.2,2021-02-05,Bengaluru,3.0,Remote
3,EMP0105,Neha Menon,Engineering,Software Engineer,30.0,Female,111847.0,NaN,2018-08-05,Jaipur,4.0,Hybrid
4,EMP0143,Hiba Menon,Sales,Sales Manager,25.0,Female,78677.0,10.9,2018-07-24,Hyderabad,4.0,Hybrid


Last 5 rows:


,Employee_ID,Employee_Name,Department,Job_Title,Age,Gender,Annual_Salary,Experience_Years,Joining_Date,City,Performance_Score,Work_Mode
152,EMP0132,Manya Rao,Marketing,Content Strategist,30.0,Female,49522.0,0.6,2021-11-25,Delhi,2.0,Remote
153,EMP0073,Karan Kapoor,Customer Success,Customer Success Manager,42.0,MALE,43331.0,1.1,2018-03-23,Jaipur,3.0,Hybrid
154,EMP0124,Saira Kapoor,Engineering,QA Engineer,33.0,Male,94812.0,7.2,2024-06-12,Jaipur,4.0,Hybrid
155,EMP0020,Sara Ali,Sales,Sales Executive,22.0,Female,51745.0,2.4,2020-11-19,NaN,2.0,Office
156,EMP0120,Saira Sheikh,Marketing,Marketing Executive,52.0,Male,107328.0,17.4,2020-11-25,Bengaluru,2.0,Remote


Random sample:


,Employee_ID,Employee_Name,Department,Job_Title,Age,Gender,Annual_Salary,Experience_Years,Joining_Date,City,Performance_Score,Work_Mode
126,EMP0045,Rohan Kumar,Sales,Business Development Executive,31.0,Male,113166.0,15.3,2023-09-05,NaN,5.0,Hybrid
45,EMP0059,Aman Sood,Engineering,QA Engineer,27.0,Male,85458.0,3.8,2023-08-05,Hyderabad,3.0,Remote
133,EMP0089,Priya Bhat,Sales,Business Development Executive,34.0,Female,91769.0,17.0,2018-10-22,Hyderabad,4.0,Remote
138,EMP0067,Vivaan Joshi,Engineering,Senior Software Engineer,41.0,Male,88676.0,8.3,2018-09-24,Hyderabad,3.0,Remote
111,EMP0113,Kabir Kumar,Operations,Operations Manager,31.0,Male,108180.0,15.6,2018-10-23,Chandigarh,5.0,Remote


Shape: (157, 12)

Columns:
['Employee_ID', 'Employee_Name', 'Department', 'Job_Title', 'Age', 'Gender', 'Annual_Salary', 'Experience_Years', 'Joining_Date', 'City', 'Performance_Score', 'Work_Mode']

Data types:
Employee_ID           object
Employee_Name         object
Department            object
Job_Title             object
Age                  float64
Gender                object
Annual_Salary        float64
Experience_Years     float64
Joining_Date          object
City                  object
Performance_Score    float64
Work_Mode             object
dtype: object


In [3]:
print("Dataset information:")
df.info()

print("\nDescriptive statistics:")
display(df.describe(include="all").T)


Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Employee_ID        157 non-null    object 
 1   Employee_Name      157 non-null    object 
 2   Department         152 non-null    object 
 3   Job_Title          157 non-null    object 
 4   Age                153 non-null    float64
 5   Gender             152 non-null    object 
 6   Annual_Salary      152 non-null    float64
 7   Experience_Years   154 non-null    float64
 8   Joining_Date       157 non-null    object 
 9   City               152 non-null    object 
 10  Performance_Score  154 non-null    float64
 11  Work_Mode          155 non-null    object 
dtypes: float64(4), object(8)
memory usage: 14.8+ KB

Descriptive statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Employee_ID,157,150,EMP0075,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Employee_Name,157,130,Nikhil Das,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Department,152,10,Marketing,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Job_Title,157,23,Data Scientist,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,153.0,NaN,NaN,NaN,38.130719,9.361661,21.0,30.0,39.0,46.0,52.0
Gender,152,5,Male,75,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Annual_Salary,152.0,NaN,NaN,NaN,89240.592105,22626.172309,43331.0,71198.5,90046.5,107328.0,162942.0
Experience_Years,154.0,NaN,NaN,NaN,8.990909,4.983874,0.6,4.45,9.35,13.2,17.4
Joining_Date,157,147,2025-08-17,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,152,13,Chandigarh,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Quantify Missing Values

In [4]:
missing_counts = df.isnull().sum()
missing_percent = (missing_counts / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing_Count": missing_counts,
    "Missing_Percentage": missing_percent
}).sort_values("Missing_Count", ascending=False)

display(missing_summary)
print("Total missing values:", int(missing_counts.sum()))


,Missing_Count,Missing_Percentage
Department,5,3.18
City,5,3.18
Annual_Salary,5,3.18
Gender,5,3.18
Age,4,2.55
Experience_Years,3,1.91
Performance_Score,3,1.91
Work_Mode,2,1.27
Job_Title,0,0.00
Employee_Name,0,0.00


Total missing values: 32


## 3. Detect Inconsistent Categorical Entries

In [5]:
categorical_columns = [
    "Department", "Gender", "City", "Work_Mode"
]

for col in categorical_columns:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).to_string())



Department:
Department
Marketing           23
Engineering         21
Customer Success    20
Finance             19
Sales               19
Operations          17
Human Resources     16
Data & Analytics    15
NaN                  5
ENGINEERING          1
engineering          1

Gender:
Gender
Male      75
Female    68
Other      7
NaN        5
female     1
MALE       1

City:
City
Chandigarh    24
Pune          23
Jaipur        18
Mumbai        15
Hyderabad     14
Srinagar      13
Bengaluru     12
Delhi         11
Chennai       10
Kolkata        9
NaN            5
delhi          1
DELHI          1
Delhi          1

Work_Mode:
Work_Mode
Remote    56
Hybrid    50
Office    47
NaN        2
REMOTE     1
remote     1


## 4. Check Numeric Data Quality and Data Types

In [6]:
numeric_columns = [
    "Age", "Annual_Salary", "Experience_Years", "Performance_Score"
]

display(df[numeric_columns].describe().T)

print("\nCurrent numeric data types:")
print(df[numeric_columns].dtypes)


,count,mean,std,min,25%,50%,75%,max
Age,153.0,38.130719,9.361661,21.0,30.00,39.00,46.0,52.0
Annual_Salary,152.0,89240.592105,22626.172309,43331.0,71198.50,90046.50,107328.0,162942.0
Experience_Years,154.0,8.990909,4.983874,0.6,4.45,9.35,13.2,17.4
Performance_Score,154.0,3.525974,0.930233,2.0,3.00,4.00,4.0,5.0



Current numeric data types:
Age                  float64
Annual_Salary        float64
Experience_Years     float64
Performance_Score    float64
dtype: object


## 5. Convert the Joining Date

In [7]:
df["Joining_Date"] = pd.to_datetime(df["Joining_Date"], errors="coerce")

print("Joining_Date dtype after conversion:", df["Joining_Date"].dtype)
print("Invalid/missing dates after conversion:", int(df["Joining_Date"].isnull().sum()))


Joining_Date dtype after conversion: datetime64[ns]
Invalid/missing dates after conversion: 0


## 6. Detect Duplicate Records

In [8]:
duplicate_rows = df.duplicated()
duplicate_count = int(duplicate_rows.sum())

print("Number of duplicate rows:", duplicate_count)

if duplicate_count > 0:
    print("\nDuplicate records:")
    display(df.loc[duplicate_rows].sort_values("Employee_ID"))


Number of duplicate rows: 7

Duplicate records:


,Employee_ID,Employee_Name,Department,Job_Title,Age,Gender,Annual_Salary,Experience_Years,Joining_Date,City,Performance_Score,Work_Mode
104,EMP0021,Riya Khan,Operations,Operations Executive,49.0,Female,71549.0,4.6,2024-09-20,Chandigarh,3.0,Hybrid
110,EMP0048,Nikhil Dar,Finance,Financial Analyst,27.0,Female,NaN,10.1,2020-06-18,Chandigarh,2.0,Office
61,EMP0075,Manya Joshi,Sales,Sales Manager,29.0,Male,92623.0,11.9,2025-08-17,Jaipur,4.0,Hybrid
55,EMP0097,Zoya Shah,Finance,Financial Analyst,41.0,Male,81531.0,4.1,2023-06-05,Mumbai,4.0,Office
156,EMP0120,Saira Sheikh,Marketing,Marketing Executive,52.0,Male,107328.0,17.4,2020-11-25,Bengaluru,2.0,Remote
100,EMP0139,Simran Sharma,Finance,Accountant,32.0,Female,110617.0,10.3,2021-08-03,Mumbai,3.0,Office
139,EMP0150,Nikhil Khan,Finance,Finance Manager,43.0,Male,129905.0,16.3,2018-04-10,Chennai,3.0,Remote


## 7. Handle Missing Numeric Values – Median Imputation

In [9]:
numeric_impute_columns = [
    "Age", "Annual_Salary", "Experience_Years", "Performance_Score"
]

numeric_imputation_values = {}

for col in numeric_impute_columns:
    median_value = df[col].median()
    numeric_imputation_values[col] = median_value
    df[col] = df[col].fillna(median_value)

print("Median values used:")
for col, value in numeric_imputation_values.items():
    print(f"{col}: {value}")


Median values used:
Age: 39.0
Annual_Salary: 90046.5
Experience_Years: 9.350000000000001
Performance_Score: 4.0


## 8. Handle Missing Categorical Values – Mode Imputation

In [10]:
categorical_impute_columns = [
    "Department", "Gender", "City", "Work_Mode"
]

categorical_imputation_values = {}

for col in categorical_impute_columns:
    mode_value = df[col].mode(dropna=True)[0]
    categorical_imputation_values[col] = mode_value
    df[col] = df[col].fillna(mode_value)

print("Mode values used:")
for col, value in categorical_imputation_values.items():
    print(f"{col}: {value}")


Mode values used:
Department: Marketing
Gender: Male
City: Chandigarh
Work_Mode: Remote


## 9. Standardize Inconsistent Categorical Values

In [11]:
# Standardize department capitalization
df["Department"] = df["Department"].str.strip().str.title()

# Standardize gender values
df["Gender"] = df["Gender"].str.strip().str.title()

# Standardize city capitalization
df["City"] = df["City"].str.strip().str.title()

# Standardize work mode capitalization
df["Work_Mode"] = df["Work_Mode"].str.strip().str.title()

print("Standardized categorical values.")
print("\nDepartments:")
print(sorted(df["Department"].unique()))
print("\nGenders:")
print(sorted(df["Gender"].unique()))
print("\nCities:")
print(sorted(df["City"].unique()))
print("\nWork modes:")
print(sorted(df["Work_Mode"].unique()))


Standardized categorical values.

Departments:


['Customer Success', 'Data & Analytics', 'Engineering', 'Finance', 'Human Resources', 'Marketing', 'Operations', 'Sales']

Genders:
['Female', 'Male', 'Other']

Cities:
['Bengaluru', 'Chandigarh', 'Chennai', 'Delhi', 'Hyderabad', 'Jaipur', 'Kolkata', 'Mumbai', 'Pune', 'Srinagar']

Work modes:
['Hybrid', 'Office', 'Remote']


## 10. Remove Duplicate Records

In [12]:
before_duplicates = len(df)

df = df.drop_duplicates().reset_index(drop=True)

after_duplicates = len(df)

print("Rows before duplicate removal:", before_duplicates)
print("Rows after duplicate removal:", after_duplicates)
print("Duplicate rows removed:", before_duplicates - after_duplicates)


Rows before duplicate removal: 157
Rows after duplicate removal: 150
Duplicate rows removed: 7


## 11. Final Data-Type Check

In [13]:
# Ensure numeric columns have numeric types
for col in numeric_impute_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.dtypes)


Employee_ID                  object
Employee_Name                object
Department                   object
Job_Title                    object
Age                         float64
Gender                       object
Annual_Salary               float64
Experience_Years            float64
Joining_Date         datetime64[ns]
City                         object
Performance_Score           float64
Work_Mode                    object
dtype: object


## 12. Verify the Cleaned Dataset

In [14]:
final_missing = df.isnull().sum()
final_duplicates = int(df.duplicated().sum())

print("Final shape:", df.shape)
print("Total remaining missing values:", int(final_missing.sum()))
print("Total remaining duplicate rows:", final_duplicates)

print("\nMissing values by column:")
display(final_missing)

print("\nFinal sample:")
display(df.head())


Final shape: (150, 12)
Total remaining missing values: 0
Total remaining duplicate rows: 0

Missing values by column:


Employee_ID          0
Employee_Name        0
Department           0
Job_Title            0
Age                  0
Gender               0
Annual_Salary        0
Experience_Years     0
Joining_Date         0
City                 0
Performance_Score    0
Work_Mode            0
dtype: int64


Final sample:


,Employee_ID,Employee_Name,Department,Job_Title,Age,Gender,Annual_Salary,Experience_Years,Joining_Date,City,Performance_Score,Work_Mode
0,EMP0098,Ananya Nair,Marketing,Data Scientist,48.0,Other,108371.0,13.20,2017-06-13,Pune,5.0,Office
1,EMP0046,Faizan Khan,Marketing,Marketing Manager,31.0,Female,108824.0,12.10,2021-09-15,Mumbai,2.0,Office
2,EMP0017,Saira Malik,Human Resources,HR Manager,28.0,Female,119400.0,13.20,2021-02-05,Bengaluru,3.0,Remote
3,EMP0105,Neha Menon,Engineering,Software Engineer,30.0,Female,111847.0,9.35,2018-08-05,Jaipur,4.0,Hybrid
4,EMP0143,Hiba Menon,Sales,Sales Manager,25.0,Female,78677.0,10.90,2018-07-24,Hyderabad,4.0,Hybrid


## 13. Before vs. After Cleaning

In [15]:
before_rows = 157
before_missing = 32
before_duplicates = 7

after_rows = len(df)
after_missing = int(df.isnull().sum().sum())
after_duplicates = int(df.duplicated().sum())

comparison = pd.DataFrame({
    "Metric": ["Rows", "Total Missing Values", "Duplicate Rows"],
    "Before Cleaning": [before_rows, before_missing, before_duplicates],
    "After Cleaning": [after_rows, after_missing, after_duplicates]
})

display(comparison)


,Metric,Before Cleaning,After Cleaning
0,Rows,157,150
1,Total Missing Values,32,0
2,Duplicate Rows,7,0


## 14. Cleaning Summary

In [16]:
print("Cleaning summary:")
print("1. Loaded and inspected the original employee dataset.")
print("2. Identified and quantified missing values using isnull().sum().")
print("3. Converted Joining_Date to datetime.")
print("4. Filled missing numeric values using median imputation.")
print("5. Filled missing categorical values using mode imputation.")
print("6. Standardized inconsistent capitalization in categorical columns.")
print("7. Detected and removed exact duplicate records using drop_duplicates().")
print("8. Rechecked missing values, duplicates, shape, and data types after cleaning.")


Cleaning summary:
1. Loaded and inspected the original employee dataset.
2. Identified and quantified missing values using isnull().sum().
3. Converted Joining_Date to datetime.
4. Filled missing numeric values using median imputation.
5. Filled missing categorical values using mode imputation.
6. Standardized inconsistent capitalization in categorical columns.
7. Detected and removed exact duplicate records using drop_duplicates().
8. Rechecked missing values, duplicates, shape, and data types after cleaning.


## 15. Conclusion

The messy company employee dataset was cleaned using Pandas. Missing numeric values were handled with median imputation, missing categorical values with mode imputation, inconsistent categorical capitalization was standardized, the joining date was converted to a proper datetime type, and duplicate records were removed. The final dataset was then verified and exported as a CSV file.

These choices follow the Day 11 notes' distinction between deletion and imputation and their examples of `isnull()`, `fillna()`, `drop_duplicates()`, and related cleaning operations. fileciteturn4file0L11-L16 fileciteturn4file0L18-L22


In [17]:
# Export the cleaned dataset
output_file = "Day11_Cleaned_Company_Employee_Dataset.csv"
df.to_csv(output_file, index=False)

print(f"Cleaned dataset saved as: {output_file}")
print("Final shape:", df.shape)


Cleaned dataset saved as: Day11_Cleaned_Company_Employee_Dataset.csv
Final shape: (150, 12)
